In [12]:
import _winapi
import json
import os
import subprocess
import tempfile
from pathlib import Path

import dcm2niix
import pandas as pd

In [2]:
# ==========================================
# 1. CONFIGURATION & DIRECTORY SETUP
# ==========================================
MASTER_CSV = "../../results/dataset_creation/clean_dedup_fmri_to_t1_master.csv"
MANIFEST_CSV = "../../results/dataset_creation/bids_conversion_manifest.csv"
FMRI_ROOT = Path("../../data/raw_fmri/ADNI")
SMRI_ROOT = Path("../../data/raw_smri/ADNI")
BIDS_ROOT = Path("../../data/raw_bids")

In [3]:
BIDS_ROOT.mkdir(parents=True, exist_ok=True)

# Create required BIDS root dataset_description.json
dataset_desc_path = BIDS_ROOT / "dataset_description.json"
if not dataset_desc_path.exists():
    dataset_desc = {
        "Name": "ADNI Resting-State fMRI and T1w BIDS Dataset",
        "BIDSVersion": "1.10.1",
        "DatasetType": "raw",
    }
    dataset_desc_path.write_text(json.dumps(dataset_desc, indent=2), encoding="utf-8")

master_df = pd.read_csv(MASTER_CSV)
print(f"Loaded master table with {len(master_df)} sessions.")

Loaded master table with 3278 sessions.


In [13]:
# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def sanitize_id(subj_str):
    """Converts ADNI ID '002_S_0413' -> 'sub-002S0413'"""
    return "sub-" + str(subj_str).replace("_", "")


def sanitize_date_ses(date_str):
    """Converts '7/27/2012' or '2012-07-27' -> 'ses-20120727'"""
    dt = pd.to_datetime(date_str, format="mixed")
    return "ses-" + dt.strftime("%Y%m%d")


def convert_dicom_to_bids(
    src_dcm_dir: Path, out_dir: Path, out_stem: str, is_bold: bool = False
):
    """
    Converts a DICOM folder directly to BIDS NIfTI (.nii.gz) + sidecar (.json) via dcm2niix.
    Skips conversion if target files already exist (resumable).
    Returns (success: bool, status_msg: str).
    """
    nii_path = out_dir / f"{out_stem}.nii.gz"
    json_path = out_dir / f"{out_stem}.json"

    if nii_path.exists() and json_path.exists():
        return True, "already_exists"

    out_dir.mkdir(parents=True, exist_ok=True)

    # Create a short temporary directory junction (e.g. C:\Users\...\Temp\dcm2niix_in)
    # so dcm2niix's internal _fullpath() never exceeds the Windows 260-char MAX_PATH limit.
    junction_dir = Path(tempfile.gettempdir()) / "dcm2niix_in"
    if os.path.lexists(junction_dir):
        os.rmdir(junction_dir)  # removes only the junction link, never target files

    _winapi.CreateJunction(str(src_dcm_dir.resolve()), str(junction_dir))
    try:
        cmd = [
            str(dcm2niix.bin),
            "-b",
            "y",  # generate BIDS JSON sidecar
            "-z",
            "y",  # gzip compress to .nii.gz
            "-f",
            out_stem,  # exact BIDS filename stem
            "-o",
            str(out_dir.resolve()),  # resolved output directory
            str(junction_dir),  # short junction path (~45 chars)
        ]
        res = subprocess.run(cmd, capture_output=True, text=True)
    finally:
        if os.path.lexists(junction_dir):
            os.rmdir(junction_dir)

    if not nii_path.exists() or not json_path.exists():
        # Check if dcm2niix appended a suffix (e.g., _e1, _a)
        produced_nii = sorted(out_dir.glob(f"{out_stem}*.nii.gz"))
        produced_json = sorted(out_dir.glob(f"{out_stem}*.json"))
        if len(produced_nii) == 1 and len(produced_json) == 1:
            produced_nii[0].rename(nii_path)
            produced_json[0].rename(json_path)
        else:
            err_lines = [
                l.strip()
                for l in (res.stderr + "\n" + res.stdout).splitlines()
                if "Error" in l or "missing" in l
            ]
            err_hint = err_lines[0][:120] if err_lines else ""
            return (
                False,
                f"dcm2niix_failed (rc={res.returncode}, nii={len(produced_nii)}) {err_hint}".strip(),
            )

    # Ensure required BIDS 'TaskName' field is present in BOLD JSON sidecar
    if is_bold and json_path.exists():
        meta = json.loads(json_path.read_text(encoding="utf-8"))
        if "TaskName" not in meta:
            meta["TaskName"] = "rest"
            json_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    return True, "converted"

In [5]:
# x_df = master_df[:5]
# x_df

In [6]:
# # Pilot test on the first 5 sessions (x_df)
# for idx, row in x_df.iterrows():
#     subject_raw = row["Subject"]
#     fmri_id = str(row["Image Data ID"])
#     t1_id = str(row["T1_Image_ID"])

#     sub_id = sanitize_id(subject_raw)
#     ses_id = sanitize_date_ses(row["Acq Date"])

#     ses_dir = BIDS_ROOT / sub_id / ses_id
#     anat_dir = ses_dir / "anat"
#     func_dir = ses_dir / "func"

#     fmri_src = list((FMRI_ROOT / subject_raw).glob(f"*/*/{fmri_id}"))
#     t1_src = list((SMRI_ROOT / subject_raw).glob(f"*/*/{t1_id}"))

#     bold_stem = f"{sub_id}_{ses_id}_task-rest_bold"
#     t1_stem = f"{sub_id}_{ses_id}_T1w"

#     ok_f, msg_f = convert_dicom_to_bids(fmri_src[0], func_dir, bold_stem, is_bold=True)
#     ok_t, msg_t = convert_dicom_to_bids(t1_src[0], anat_dir, t1_stem, is_bold=False)
#     print(f"{sub_id} {ses_id} | BOLD: {msg_f} | T1w: {msg_t}")


In [14]:
# ==========================================
# 3. FULL BIDS CONVERSION LOOP (dcm2niix)
# ==========================================
manifest = []
missing_fmri = 0
missing_smri = 0
total = len(master_df)

for idx, row in master_df.iterrows():
    subject_raw = row["Subject"]  # e.g., '002_S_0413'
    fmri_id = str(row["Image Data ID"])  # e.g., 'I322442'
    t1_id = str(row["T1_Image_ID"])  # e.g., 'I322438'

    fmri_date = row["Acq Date"]
    t1_date = row["T1_Study_Date"]

    sub_id = sanitize_id(subject_raw)
    ses_id = sanitize_date_ses(fmri_date)

    ses_dir = BIDS_ROOT / sub_id / ses_id
    anat_dir = ses_dir / "anat"
    func_dir = ses_dir / "func"

    # --- A. LOCATE & CONVERT fMRI DICOMS ---
    fmri_src_folders = list((FMRI_ROOT / subject_raw).glob(f"*/*/{fmri_id}"))
    if fmri_src_folders:
        bold_stem = f"{sub_id}_{ses_id}_task-rest_bold"
        fmri_success, fmri_status = convert_dicom_to_bids(
            fmri_src_folders[0], func_dir, bold_stem, is_bold=True
        )
    else:
        fmri_success, fmri_status = False, "missing_source_folder"
        missing_fmri += 1

    # --- B. LOCATE & CONVERT sMRI (T1w) DICOMS ---
    t1_src_folders = list((SMRI_ROOT / subject_raw).glob(f"*/*/{t1_id}"))
    if t1_src_folders:
        t1_stem = f"{sub_id}_{ses_id}_T1w"
        t1_success, t1_status = convert_dicom_to_bids(
            t1_src_folders[0], anat_dir, t1_stem, is_bold=False
        )
    else:
        t1_success, t1_status = False, "missing_source_folder"
        missing_smri += 1

    manifest.append(
        {
            "Subject": subject_raw,
            "BIDS_Subject": sub_id,
            "BIDS_Session": ses_id,
            "fMRI_Image_ID": fmri_id,
            "fMRI_Acq_Date": fmri_date,
            "fMRI_Converted": fmri_success,
            "fMRI_Status": fmri_status,
            "T1_Image_ID": t1_id,
            "T1_Study_Date": t1_date,
            "T1_Converted": t1_success,
            "T1_Status": t1_status,
        }
    )

    if (idx + 1) % 100 == 0 or (idx + 1) == total:
        print(f"[{idx + 1}/{total}] processed (latest: {sub_id} {ses_id})")

KeyboardInterrupt: 

In [8]:
# ==========================================
# 4. SAVE MANIFEST & REPORT
# ==========================================
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print("\n--- BIDS Conversion Complete ---")
print(f"Total sessions processed:        {len(master_df)}")
print(f"Successfully converted fMRI:     {manifest_df['fMRI_Converted'].sum()}")
print(f"Successfully converted T1w:      {manifest_df['T1_Converted'].sum()}")
print(f"Missing fMRI folders on disk:    {missing_fmri}")
print(f"Missing T1w folders on disk:     {missing_smri}")
print(f"Manifest saved to:               {MANIFEST_CSV}")


--- BIDS Conversion Complete ---
Total sessions processed:        3278
Successfully converted fMRI:     3170
Successfully converted T1w:      3205
Missing fMRI folders on disk:    0
Missing T1w folders on disk:     0
Manifest saved to:               ../../results/dataset_creation/bids_conversion_manifest.csv


In [ ]:
# ==========================================
# 5. RETRY ONLY LONG-PATH FAILED RECORDS (rc=2)
# ==========================================
manifest_df = pd.read_csv(MANIFEST_CSV)

# Only retry scans that failed due to Windows MAX_PATH ('Unable to find any DICOM images')
# (Scans with rc=1 already had their DICOMs read and failed due to missing/corrupted slices)
retry_f_mask = (~manifest_df["fMRI_Converted"]) & manifest_df[
    "fMRI_Status"
].str.contains("Unable to find any DICOM", na=False)
retry_t_mask = (~manifest_df["T1_Converted"]) & manifest_df["T1_Status"].str.contains(
    "Unable to find any DICOM", na=False
)
failed_indices = manifest_df[retry_f_mask | retry_t_mask].index.tolist()

print(
    f"Retrying {len(failed_indices)} sessions ({retry_f_mask.sum()} fMRI, {retry_t_mask.sum()} T1w) "
    f"that failed due to MAX_PATH..."
)
recovered_fmri = 0
recovered_t1 = 0

for count, idx in enumerate(failed_indices, start=1):
    row = manifest_df.loc[idx]
    subject_raw = row["Subject"]
    sub_id = row["BIDS_Subject"]
    ses_id = row["BIDS_Session"]
    fmri_id = str(row["fMRI_Image_ID"])
    t1_id = str(row["T1_Image_ID"])

    ses_dir = BIDS_ROOT / sub_id / ses_id
    anat_dir = ses_dir / "anat"
    func_dir = ses_dir / "func"

    # Retry fMRI only if it failed with MAX_PATH
    if retry_f_mask.loc[idx]:
        fmri_src_folders = list((FMRI_ROOT / subject_raw).glob(f"*/*/{fmri_id}"))
        if fmri_src_folders:
            bold_stem = f"{sub_id}_{ses_id}_task-rest_bold"
            ok_f, status_f = convert_dicom_to_bids(
                fmri_src_folders[0], func_dir, bold_stem, is_bold=True
            )
            manifest_df.at[idx, "fMRI_Converted"] = ok_f
            manifest_df.at[idx, "fMRI_Status"] = status_f
            if ok_f:
                recovered_fmri += 1

    # Retry T1w only if it failed with MAX_PATH
    if retry_t_mask.loc[idx]:
        t1_src_folders = list((SMRI_ROOT / subject_raw).glob(f"*/*/{t1_id}"))
        if t1_src_folders:
            t1_stem = f"{sub_id}_{ses_id}_T1w"
            ok_t, status_t = convert_dicom_to_bids(
                t1_src_folders[0], anat_dir, t1_stem, is_bold=False
            )
            manifest_df.at[idx, "T1_Converted"] = ok_t
            manifest_df.at[idx, "T1_Status"] = status_t
            if ok_t:
                recovered_t1 += 1

    if count % 10 == 0 or count == len(failed_indices):
        print(
            f"[{count}/{len(failed_indices)}] retried | "
            f"Recovered so far -> fMRI: +{recovered_fmri}, T1w: +{recovered_t1}"
        )

manifest_df.to_csv(MANIFEST_CSV, index=False)
both_ok = (manifest_df["fMRI_Converted"] & manifest_df["T1_Converted"]).sum()
print("\n--- Retry Complete ---")
print(f"Newly recovered fMRI:            +{recovered_fmri}")
print(f"Newly recovered T1w:             +{recovered_t1}")
print(
    f"Total fMRI converted:            {manifest_df['fMRI_Converted'].sum()} / {len(manifest_df)}"
)
print(
    f"Total T1w converted:             {manifest_df['T1_Converted'].sum()} / {len(manifest_df)}"
)
print(f"Total complete pairs (both ok):  {both_ok} / {len(manifest_df)}")
print(f"Remaining failed fMRI:           {(~manifest_df['fMRI_Converted']).sum()}")
print(f"Remaining failed T1w:            {(~manifest_df['T1_Converted']).sum()}")

Retrying 98 sessions (73 fMRI, 72 T1w) that failed due to MAX_PATH...
[10/98] retried | Recovered so far -> fMRI: +0, T1w: +10
[20/98] retried | Recovered so far -> fMRI: +0, T1w: +20
[30/98] retried | Recovered so far -> fMRI: +5, T1w: +25
[40/98] retried | Recovered so far -> fMRI: +15, T1w: +25
[50/98] retried | Recovered so far -> fMRI: +25, T1w: +33
[60/98] retried | Recovered so far -> fMRI: +35, T1w: +43
[70/98] retried | Recovered so far -> fMRI: +45, T1w: +53
[80/98] retried | Recovered so far -> fMRI: +55, T1w: +63
[90/98] retried | Recovered so far -> fMRI: +65, T1w: +72
[98/98] retried | Recovered so far -> fMRI: +73, T1w: +72

--- Retry Complete ---
Newly recovered fMRI:            +73
Newly recovered T1w:             +72
Total fMRI converted:            3243 / 3278
Total T1w converted:             3277 / 3278
Total complete pairs (both ok):  3242 / 3278
Remaining failed fMRI:           35
Remaining failed T1w:            1
